In [5]:
import pm4py
import json
import pandas as pd
from collections import defaultdict

# Load OCEL
ocel = pm4py.read_ocel2_json("commitizen_task_objects.json")

with open("commitizen_task_objects.json") as f:
    data = json.load(f)

print("Events:", len(ocel.events))
print("Objects:", len(ocel.objects))
print("Object types:", ocel.objects[ocel.object_type_column].value_counts().to_dict())

Events: 21488
Objects: 6534
Object types: {'commit': 2765, 'task': 1721, 'issue': 1459, 'user': 589}


In [6]:
# Build a lookup: object_id -> attributes dict
# Note: store OCEL object type as "obj_type" to avoid collision
# with the GitHub "type" attribute ("User" / "Bot")
obj_attrs = {}
for obj in data["objects"]:
    attrs = {a["name"]: a["value"] for a in obj.get("attributes", [])}
    obj_attrs[obj["id"]] = {"obj_type": obj["type"], **attrs}

# Identify all user objects
users = [
    oid for oid, info in obj_attrs.items()
    if info["obj_type"] == "user"
]
print(f"Total user objects: {len(users)}")

# Identify bots immediately (GitHub "type" attribute == "Bot")
bots = [u for u in users if obj_attrs[u].get("type", "") == "Bot"]
print(f"Bots: {len(bots)}")

Total user objects: 589
Bots: 6


In [7]:
# Build resource profiles
# For each user: activity distribution, task semantic class distribution, commit count

TASK_CLASSES = [
    "feature_development",
    "defect_resolution",
    "code_restructuring",
    "documentation",
    "quality_assurance",
    "deployment_engineering",
    "build_engineering",
    "maintenance",
    "performance_improvement",
    "formatting",
]

profiles = {}  # user_id -> profile dict

for user_id in users:
    # Events this user participated in
    user_events = ocel.relations[
        ocel.relations["ocel:oid"] == user_id
    ]["ocel:eid"].unique()

    if len(user_events) == 0:
        profiles[user_id] = {
            "event_count": 0,
            "commit_count": 0,
            "task_counts": {},
            "activity_counts": {},
        }
        continue

    # Activity distribution
    activity_counts = (
        ocel.events.loc[
            ocel.events.index.isin(user_events),
            "ocel:activity"
        ]
        .value_counts()
        .to_dict()
    )

    # All objects co-occurring with this user in those events
    co_objects = ocel.relations[
        ocel.relations["ocel:eid"].isin(user_events)
    ]

    # Commits linked to this user
    commit_ids = co_objects[
        co_objects["ocel:type"] == "commit"
    ]["ocel:oid"].unique()

    # Task semantic class distribution via commit attributes
    task_class_counts = defaultdict(int)
    for cid in commit_ids:
        cls = obj_attrs.get(cid, {}).get("task_semantic_class")
        if cls:
            task_class_counts[cls] += 1

    profiles[user_id] = {
        "event_count": len(user_events),
        "commit_count": len(commit_ids),
        "task_counts": dict(task_class_counts),
        "activity_counts": activity_counts,
    }

# Summarise
# Note: use `or` instead of .get() default to handle None values
# (158 users are ghost/deleted accounts with all attributes set to None)
df_profiles = pd.DataFrame([
    {
        "user_id": uid,
        "login": obj_attrs[uid].get("login") or uid,
        "github_type": obj_attrs[uid].get("type") or "Unknown",
        "event_count": p["event_count"],
        "commit_count": p["commit_count"],
        **{f"task_{k}": p["task_counts"].get(k, 0) for k in TASK_CLASSES},
    }
    for uid, p in profiles.items()
]).set_index("user_id")

print(df_profiles.shape)
df_profiles.sort_values("commit_count", ascending=False).head(20)

(589, 14)


,login,github_type,event_count,commit_count,task_feature_development,task_defect_resolution,task_code_restructuring,task_documentation,task_quality_assurance,task_deployment_engineering,task_build_engineering,task_maintenance,task_performance_improvement,task_formatting
user_id,,,,,,,,,,,,,,
52,52,Unknown,919,919,50,66,209,93,104,54,55,1,2,72
2652,2652,Unknown,457,457,3,7,2,8,1,1,1,1,0,0
265,265,Unknown,361,361,3,20,149,53,26,5,22,0,10,56
61,61,Unknown,251,251,26,38,14,71,19,20,9,1,0,7
62262,62262,Unknown,69,69,9,4,0,3,6,2,0,1,0,1
18478,18478,Unknown,37,37,1,2,0,1,3,1,25,0,0,0
48373,48373,Unknown,36,36,0,0,22,0,0,0,0,0,0,0
16562,16562,Unknown,33,33,7,8,1,0,4,2,4,0,0,2
65498,65498,Unknown,24,24,0,0,0,0,0,0,0,0,0,2


In [8]:
# Role assignment — rule-based
#
# Rules applied in order (first match wins):
#   bot              github_type == "Bot"
#   issue_reporter   commit_count == 0  (participates but never commits)
#   maintainer       commit_count >= MAINTAINER_THRESHOLD and task diversity >= 3 classes
#   devops_engineer  majority class in {deployment_engineering, build_engineering}
#   quality_engineer majority class in {quality_assurance, defect_resolution}
#   technical_writer majority class == documentation
#   feature_developer majority class in {feature_development, code_restructuring}
#   contributor      everything else (low volume or single contribution)

MAINTAINER_THRESHOLD = 20  # minimum commits to be considered a maintainer
DEVOPS_CLASSES = {"deployment_engineering", "build_engineering"}
QA_CLASSES = {"quality_assurance", "defect_resolution"}
FEATURE_CLASSES = {"feature_development", "code_restructuring"}


def majority_class(task_counts: dict) -> str | None:
    """Return the task class with the highest count, or None if no tasks."""
    if not task_counts:
        return None
    return max(task_counts, key=task_counts.get)


def majority_group(task_counts: dict, group: set) -> bool:
    """True if the combined count of classes in `group` is the plurality."""
    if not task_counts:
        return False
    group_total = sum(task_counts.get(c, 0) for c in group)
    total = sum(task_counts.values())
    return total > 0 and (group_total / total) >= 0.5


def assign_role(user_id: str, profile: dict, github_type: str) -> str:
    if github_type == "Bot":
        return "bot"

    commit_count = profile["commit_count"]
    task_counts = profile["task_counts"]
    n_classes = sum(1 for v in task_counts.values() if v > 0)

    if commit_count == 0:
        return "issue_reporter"

    if commit_count >= MAINTAINER_THRESHOLD and n_classes >= 3:
        return "maintainer"

    if majority_group(task_counts, DEVOPS_CLASSES):
        return "devops_engineer"

    if majority_group(task_counts, QA_CLASSES):
        return "quality_engineer"

    mc = majority_class(task_counts)
    if mc == "documentation":
        return "technical_writer"

    if majority_group(task_counts, FEATURE_CLASSES):
        return "feature_developer"

    return "contributor"


roles = {
    uid: assign_role(uid, profiles[uid], obj_attrs[uid].get("type") or "Unknown")
    for uid in users
}

role_series = pd.Series(roles, name="role")
print(role_series.value_counts())

role
issue_reporter       425
contributor           66
quality_engineer      37
technical_writer      23
feature_developer     20
maintainer             8
bot                    6
devops_engineer        4
Name: count, dtype: int64


In [9]:
# Inspect role assignments for top contributors
df_profiles["role"] = role_series

print("=== Maintainers ===")
print(df_profiles[df_profiles["role"] == "maintainer"][["login", "commit_count", "event_count"]].sort_values("commit_count", ascending=False).to_string())

print("\n=== Feature Developers (sample) ===")
print(df_profiles[df_profiles["role"] == "feature_developer"][["login", "commit_count"]].sort_values("commit_count", ascending=False).head(10).to_string())

print("\n=== Bots ===")
print(df_profiles[df_profiles["role"] == "bot"][["login", "event_count"]].to_string())

=== Maintainers ===
         login  commit_count  event_count
user_id                                  
52          52           919          919
2652      2652           457          457
265        265           361          361
61          61           251          251
62262    62262            69           69
18478    18478            37           37
16562    16562            33           33
38798    38798            22           22

=== Feature Developers (sample) ===
         login  commit_count
user_id                     
48373    48373            36
67070    67070             9
55508    55508             9
62707    62707             8
49912    49912             7
70778    70778             5
82892    82892             4
4097      4097             4
19274    19274             3
81326    81326             2

=== Bots ===
                           login  event_count
user_id                                      
94           github-actions[bot]          938
8753                   

In [10]:
# Write role annotations back into the OCEL JSON
# Each user object gets a new attribute: role

import copy
from datetime import datetime, timezone

output_data = copy.deepcopy(data)

# Ensure 'role' is declared in the user objectType attributes schema
for ot in output_data["objectTypes"]:
    if ot["name"] == "user":
        existing_attr_names = [a["name"] for a in ot.get("attributes", [])]
        if "role" not in existing_attr_names:
            ot.setdefault("attributes", []).append({"name": "role", "type": "string"})
        break

# Stamp to use for the annotation time
annotation_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

annotated = 0
for obj in output_data["objects"]:
    if obj["type"] == "user" and obj["id"] in roles:
        obj.setdefault("attributes", []).append({
            "name": "role",
            "value": roles[obj["id"]],
            "time": annotation_time,
        })
        annotated += 1

print(f"Annotated {annotated} user objects with role attribute.")

output_path = "commitizen_role_annotated.json"
with open(output_path, "w") as f:
    json.dump(output_data, f)

print(f"Saved to {output_path}")

Annotated 589 user objects with role attribute.
Saved to commitizen_role_annotated.json


In [11]:
# Verify: reload and spot-check
ocel_annotated = pm4py.read_ocel2_json(output_path)

print("Objects in annotated OCEL:", len(ocel_annotated.objects))

with open(output_path) as f:
    ann_data = json.load(f)

# Show role distribution from the raw JSON
role_counts = defaultdict(int)
for obj in ann_data["objects"]:
    if obj["type"] == "user":
        role = next(
            (a["value"] for a in obj.get("attributes", []) if a["name"] == "role"),
            "(none)"
        )
        role_counts[role] += 1

print("\nRole distribution in annotated OCEL:")
for role, count in sorted(role_counts.items(), key=lambda x: -x[1]):
    print(f"  {role:25s} {count}")

Objects in annotated OCEL: 6534

Role distribution in annotated OCEL:
  issue_reporter            425
  contributor               66
  quality_engineer          37
  technical_writer          23
  feature_developer         20
  maintainer                8
  bot                       6
  devops_engineer           4
